In [9]:
import sys
import time
from dataclasses import dataclass

import torch

# ---------------------------------------------------------
# Patch: mache torch.load wieder kompatibel mit alten Checkpoints
# (PyTorch 2.6 setzt sonst weights_only=True und bricht)
# ---------------------------------------------------------
def patch_torch_load_for_neuspell() -> None:
    # Wenn schon gepatcht → nichts tun
    if getattr(torch.load, "__name__", "") == "_torch_load_legacy":
        return

    orig_torch_load = torch.load

    def _torch_load_legacy(*args, **kwargs):
        if "weights_only" not in kwargs:
            kwargs["weights_only"] = False
        return orig_torch_load(*args, **kwargs)

    torch.load = _torch_load_legacy


patch_torch_load_for_neuspell()

# ---------------------------------------------------------
# NeuSpell SCLSTM
# ---------------------------------------------------------
from neuspell import SclstmChecker

CHECKPOINT_DIR = "/home/vietc/projects/search-engine/src/backend/models/neuspell-scrnn-probwordnoise"

In [10]:
@dataclass
class SpellCorrector:
    checker: SclstmChecker

    @classmethod
    def load(cls) -> "SpellCorrector":
        print("load neuspell SCLSTM checker…")
        checker = SclstmChecker()
        checker.from_pretrained(CHECKPOINT_DIR)
        print("model loaded.\n")
        return cls(checker=checker)

    def correct(self, text: str) -> str:
        return self.checker.correct(text)


In [ ]:
def repl(corrector: SpellCorrector) -> None:
    print("NeuSpell SCLSTM Spell-Correction Test-REPL")
    print("Gib eine Query ein (leer + Enter zum Beenden).\n")

    while True:
        try:
            query = input("Query> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye.")
            return

        if not query:
            print("Beende.")
            return

        start = time.perf_counter()
        corrected = corrector.correct(query)
        elapsed_ms = (time.perf_counter() - start) * 1000

        print(f"queue: {query}")
        print(f"corrected: {corrected}")
        print(f"time: {elapsed_ms:.2f} ms")
        print("-" * 40)

In [12]:
def main() -> None:
    corrector = SpellCorrector.load()
    repl(corrector)


if __name__ == "__main__":
    sys.exit(main())

load neuspell SCLSTM checker…
loading vocab from path:/home/vietc/projects/search-engine/src/backend/models/neuspell-scrnn-probwordnoise/vocab.pkl
initializing model
SCLSTM(
  (lstmmodule): LSTM(294, 512, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (dense): Linear(in_features=1024, out_features=100002, bias=True)
  (criterion): CrossEntropyLoss()
)
112111266
loading pretrained weights from path:/home/vietc/projects/search-engine/src/backend/models/neuspell-scrnn-probwordnoise
Loading model params from checkpoint dir: /home/vietc/projects/search-engine/src/backend/models/neuspell-scrnn-probwordnoise
model loaded.

NeuSpell SCLSTM Spell-Correction Test-REPL
Gib eine Query ein (leer + Enter zum Beenden).

###############################################
data size: 1
total inference time for this data is: 0.041763 secs
Original : wintre
Korrigiert: winter
Dauer    : 42.38 ms
----------------------------------------
Beende.


SystemExit: 